In [12]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# 1. Load the merged dataset from excel
xls = pd.ExcelFile('merged.xlsx')
df = pd.read_excel(xls, xls.sheet_names[0])

df

,Client ID,TYPE,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,YEAR,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,YEAR_cogs,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021,3,4,4,5,8,2021,148180,17321,25562,41201
1,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021,3,4,4,5,8,2022,197313,23604,32875,50085
2,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021,3,4,4,5,8,2023,349899,51332,71695,89089
3,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021,3,4,4,5,8,2024,268821,40969,56452,76545
4,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021,3,4,4,5,8,2025,138115,14516,20889,28968
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1524,P3-027,Private,2025-01-04 00:00:00,> 200,Info Tech,Malaysian,2025,4,5,5,3,9,2025,172191,17127,35088,52215
1525,P3-028,Private,2025-01-14 00:00:00,50 ~ 200,Info Tech,India,2025,4,5,5,5,9,2025,220048,15847,45103,91424
1526,P3-029,Private,2025-04-04 00:00:00,> 200,Healthcare,Malaysia,2025,5,2,4,5,9,2025,279839,22351,45673,70801
1527,P3-030,Private,2025-05-04 00:00:00,50 ~ 200,Healthcare,Indonesia,2025,3,3,3,3,6,2025,146829,14505,30602,43338


In [30]:
# Copy dataframe
plot_df = df.copy()

# Create Overall Satisfaction
plot_df["Overall Satisfaction"] = (
    plot_df[
        [
            "PRESALES AND PARTNERSHIP",
            "TECHNICAL EXPERTISE",
            "PROJECT DELIVERY",
            "POST-SALES SUPPORT",
        ]
    ].mean(axis=1)
)

plot_df["COGS"] = (
    plot_df["HARDWARE"] + plot_df["SOFTWARE"] + plot_df["MANPOWER"]
)

# Gross Profit and Gross Margin
plot_df["Gross Profit"] = (
    plot_df["REVENUE"]
    - plot_df["COGS"]
)

plot_df["Gross Margin"] = (
    plot_df["Gross Profit"] / plot_df["REVENUE"] * 100
)

# NPS Category
plot_df["NPS Category"] = np.select(
    [
        plot_df["NPS RATING"] >= 9,
        plot_df["NPS RATING"] >= 7
    ],
    [
        "Promoter",
        "Passive"
    ],
    default="Detractor"
)

# Add small horizontal jitter
np.random.seed(42)

plot_df["Satisfaction Jitter"] = (
    plot_df["Overall Satisfaction"]
    + np.random.uniform(-0.1, 0.1, len(plot_df))
).clip(lower=1, upper=5)


In [33]:
plot_df["Bubble Size"] = np.sqrt(plot_df["REVENUE"])

fig = px.scatter(
    plot_df,
    x="Satisfaction Jitter",
    y="REVENUE",
    size="Bubble Size",
    color="NPS Category",
    hover_name="Client ID",
    hover_data={
        "Overall Satisfaction": ":.2f",
        "Gross Profit": ":,.0f",
        "Gross Margin": ":.1f",
        "REVENUE": ":,.0f",
        "Satisfaction Jitter": False
    },
    opacity=0.6,
    trendline="ols",
    title="Client Satisfaction vs Gross Margin"
)

fig.update_layout(
    xaxis_title="Overall Satisfaction",
    yaxis_title="Gross Margin (%)",
    legend_title="NPS Category",
    template="plotly_white"
)

fig.update_xaxes(
    tickvals=np.arange(1, 5.25, 0.25),
    range=[1.75, 5.2]
)

fig.show()